In [5]:
import numpy as np
import random
from scipy.spatial import distance_matrix

# Defining city coordinates
cities = {
    'A': (100, 300),
    'B': (200, 130),
    'C': (300, 500),
    'D': (500, 390),
    'E': (700, 300),
    'F': (900, 600),
    'G': (800, 950),
    'H': (600, 560),
    'I': (350, 550),
    'J': (270, 350)
}

# Extracting coordinates into an array
coords = np.array(list(cities.values()))

# Calculate distance matrix
dist_matrix = distance_matrix(coords, coords)

# Helper function to calculate the total distance of a given path
def total_distance(path):
    dist = 0
    for i in range(len(path) - 1):
        dist += dist_matrix[path[i], path[i + 1]]
    dist += dist_matrix[path[-1], path[0]]  # return to the start
    return dist

# Genetic Algorithm parameters
population_size = 100
generations = 500
mutation_rate = 0.2
tournament_size = 2

# Create a population of random tours
def create_population(size, num_cities):
    return [random.sample(range(num_cities), num_cities) for _ in range(size)]

# Perform crossover (ordered crossover)
def crossover(parent1, parent2):
    start, end = sorted(random.sample(range(len(parent1)), 2))
    child = [-1] * len(parent1)
    child[start:end] = parent1[start:end]
    p2_idx = 0
    for i in range(len(parent2)):
        if parent2[i] not in child:
            while child[p2_idx] != -1:
                p2_idx += 1
            child[p2_idx] = parent2[i]
    return child

# Perform mutation (swap mutation)
def mutate(tour):
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(tour)), 2)
        tour[i], tour[j] = tour[j], tour[i]

# Tournament selection
def tournament_selection(population, fitnesses):
    tournament = random.sample(list(zip(population, fitnesses)), tournament_size)
    tournament.sort(key=lambda x: x[1])
    return tournament[0][0]

# Run the Genetic Algorithm
def genetic_algorithm():
    num_cities = len(cities)
    population = create_population(population_size, num_cities)
    best_distance = float('inf')
    best_tour = None

    for generation in range(generations):
        fitnesses = [total_distance(tour) for tour in population]
        new_population = []

        for _ in range(population_size // 2):
            # Selection
            parent1 = tournament_selection(population, fitnesses)
            parent2 = tournament_selection(population, fitnesses)

            # Crossover
            child1 = crossover(parent1, parent2)
            child2 = crossover(parent2, parent1)

            # Mutation
            mutate(child1)
            mutate(child2)

            new_population.extend([child1, child2])

        population = new_population

        # Track the best solution
        for tour in population:
            dist = total_distance(tour)
            if dist < best_distance:
                best_distance = dist
                best_tour = tour

    return best_tour, best_distance

# Run the algorithm
best_tour, best_distance = genetic_algorithm()

# Map city indices back to city names
best_tour_named = [list(cities.keys())[i] for i in best_tour]

best_tour_named, best_distance


(['D', 'B', 'A', 'J', 'C', 'I', 'H', 'G', 'F', 'E'], 2627.4710131370725)